# Computation Thinking 정리 노트

이 노트는 **컴퓨터처럼 사고하기(논리/증명/재귀/복잡도)** 파트를 한 번에 복습할 수 있게 정리한 자료입니다.  
그림(스크린샷)은 원본 강의 자료를 그대로 붙여두었고, 필요한 부분은 코드로도 직접 확인할 수 있게 했습니다.

---

## 오늘의 목표
- **명제/논리 연산**을 “진리표”로 정확히 다룰 수 있다.
- **조건명제(p→q)** 가 언제 거짓이 되는지, **대우**가 왜 중요한지 설명할 수 있다.
- 간단한 **증명 아이디어(Trivial / Vacuous)** 를 상황에 맞게 적용할 수 있다.
- **재귀**의 핵심(종료 조건, 호출/복귀 흐름)을 말로 설명하고 코드로 구현할 수 있다.
- 자주 나오는 **점화식(재귀식) 시간복잡도**를 빠르게 판별할 수 있다.


## 전체 흐름(알고리즘 응용 파트로 이어짐)

- 컴퓨터처럼 사고하기  
  - 논리적 사고  
  - 진법과 비트 연산(추후)

- 알고리즘 응용  
  - 완전 검색 / 그리디 / 분할정복 / 백트래킹  
  - 그래프(기초 이론 → BFS/DFS 완전탐색 → 최단거리/최소비용)

> 쉽게 말하면, **코드를 빨리 치는 연습**보다 먼저  
> “이게 참인지 거짓인지”, “어떤 경우를 검사해야 하는지”, “얼마나 오래 걸리는지”를 **컴퓨터 눈으로 보는 연습**이에요.


## 1. 명제와 논리 연산

### 명제(Proposition)
- **참(True) / 거짓(False)** 으로 판별 가능한 문장
  - 예) `2는 짝수이다` → 참  
  - 예) `3은 짝수이다` → 거짓  
  - 예) `강의는 재밌다` → 사람마다 달라서(판별 기준이 없어서) 보통 명제로 보지 않음

### 기본 연산(부정/그리고/또는 등)
아래는 가장 많이 쓰는 논리 연산들입니다.  
(표는 “진리표” 형태로 보면서 외우는 게 가장 빠릅니다.)


#### 부정(~p) / AND(∧) / OR(∨) / XOR(⊕) / 조건(→) / 동치(↔)

![](<스크린샷 2026-03-03 141930.png>)
![](<스크린샷 2026-03-03 142103.png>)
![](<스크린샷 2026-03-03 142242.png>)
![](<스크린샷 2026-03-03 142400.png>)
![](<스크린샷 2026-03-03 142516.png>)

> 팁: 조건명제(p→q)는 **p가 참인데 q가 거짓일 때만 거짓**이에요.  
> 나머지는 전부 참이라서 처음엔 어색한데, 이 규칙을 잡으면 문제가 확 쉬워집니다.


In [3]:
# (확인용) 파이썬으로 진리표를 직접 찍어보기
from itertools import product

def implies(p, q):
    return (not p) or q

def iff(p, q):
    return p == q

def xor(p, q):
    return (p and (not q)) or ((not p) and q)

rows = []
for p, q in product([True, False], repeat=2):
    rows.append({
        "p": p, "q": q,
        "~p": (not p),
        "p∧q": (p and q),
        "p∨q": (p or q),
        "p⊕q": xor(p, q),
        "p→q": implies(p, q),
        "p↔q": iff(p, q),
    })

# pandas가 있으면 표로, 없으면 그냥 출력
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    df
except Exception:
    for r in rows:
        print(r)


{'p': True, 'q': True, '~p': False, 'p∧q': True, 'p∨q': True, 'p⊕q': False, 'p→q': True, 'p↔q': True}
{'p': True, 'q': False, '~p': False, 'p∧q': False, 'p∨q': True, 'p⊕q': True, 'p→q': False, 'p↔q': False}
{'p': False, 'q': True, '~p': True, 'p∧q': False, 'p∨q': True, 'p⊕q': True, 'p→q': True, 'p↔q': False}
{'p': False, 'q': False, '~p': True, 'p∧q': False, 'p∨q': False, 'p⊕q': False, 'p→q': True, 'p↔q': True}


## 2. 조건명제(p→q) 제대로 이해하기

### p→q 가 거짓이 되는 유일한 경우
- **p가 참인데 q가 거짓일 때만 거짓**
- 그 외(특히 p가 거짓인 경우)는 전부 참

> 쉽게 말하면, “만약 숙제를 하면(=p) 게임하게 해줄게(=q)”라는 약속을 했는데  
> **숙제 했는데 게임 못 하면** 약속 위반(거짓)  
> 숙제를 안 했으면 애초에 약속 조건이 성립하지 않아서(판정 대상이 아니라서) “거짓”으로 보지 않는다고 생각하면 편해요.

### 역/이/대우
- **역(converse)**: q→p  
- **이(inverse)**: ~p→~q  
- **대우(contrapositive)**: ~q→~p  

그리고 중요한 결론:
- **원래 명제(p→q)와 ‘대우(~q→~p)’는 항상 같은 참/거짓을 가짐(동치)**
- 역/이는 일반적으로 동치가 아님

![](<스크린샷 2026-03-03 142652.png>)


In [1]:
# (확인용) p→q 와 ~q→~p 가 항상 같은지 전수검사
from itertools import product

def implies(p, q):
    return (not p) or q

ok = True
for p, q in product([True, False], repeat=2):
    left = implies(p, q)
    right = implies((not q), (not p))  # ~q → ~p
    if left != right:
        ok = False
        print("반례 발견:", p, q, left, right)

print("대우는 항상 동치인가?", ok)


대우는 항상 동치인가? True


## 3. 연습 문제로 감 잡기

### 문제 1) 명제식으로 쓰고 참/거짓 판단
![](<스크린샷 2026-03-03 143314.png>)

핵심 포인트:
- p→q에서 **p가 거짓이면 전체는 참**이 될 수 있음(“조건이 안 맞으면” 약속 위반이 아님)
- 어떤 문제는 대우(~q→~p)로 보면 바로 끝남


### 문제 2) p→q 가 거짓일 때, 다른 식들의 참/거짓
![](<스크린샷 2026-03-03 143328.png>)

핵심 포인트:
- p→q 가 거짓이라는 건 **(p=True, q=False)로 고정**된다는 뜻  
- 그러면 나머지 식은 그 값만 넣고 계산하면 됨


### 문제 3) 역/이/대우 직접 써보기
![](<스크린샷 2026-03-03 143424.png>)

정리:
- 원래 명제와 **대우**는 항상 함께 맞고 함께 틀림
- 역/이는 그럴 수도 있고 아닐 수도 있음


## 4. 카드 문제(‘반례 찾기’ 사고)

> 사실: 모든 카드의 한쪽에는 알파벳, 다른 쪽에는 숫자  
> 주장: “한쪽이 D이면 반대쪽은 3이다.”

![](<스크린샷 2026-03-03 140636.png>)

### 반드시 뒤집어야 하는 카드
- **D**: 정말 반대쪽이 3인지 확인해야 함(규칙 위반 가능)
- **7**: 만약 반대쪽이 D라면 “D인데 3이 아님”이 되어 규칙 위반

따라서 답은 **2장(D와 7)**.

> 쉽게 말하면, 규칙을 “증명”하려고 하기보다 **규칙을 깨는 반례가 있는지** 찾는 문제예요.  
> 조건명제는 “p면 q”이므로, 반례는 항상 “p인데 q가 아니다” 형태로 생깁니다.


## 5. 진리표 만들기(연산 순서 익히기)

![](<스크린샷 2026-03-03 143524.png>)
![](<스크린샷 2026-03-03 143540.png>)

### 풀이 팁
- 진리표는 항상 **기본 열(p, q, r...) → 중간 열(~p, p∧q...) → 최종 식** 순서로 채우면 실수가 줄어듭니다.
- 조건(→)은 “거짓 되는 유일한 줄”만 정확히 기억하면 빨라요.


## 6. 증명 아이디어 2개: Trivial / Vacuous

### Trivial Proof
- ∀x, P(x)→Q(x)를 보이려는데 **Q(x)가 항상 참**이면, P(x)가 뭐든 상관없이 전체는 참

![](<스크린샷 2026-03-03 143649.png>)

### Vacuous Proof
- ∀x, P(x)→Q(x)를 보이려는데 **P(x)가 항상 거짓**이면, 역시 전체는 참

![](<스크린샷 2026-03-03 143703.png>)

> 쉽게 말하면  
> - Trivial: “결론이 항상 맞는 말이면” 조건을 만족하든 말든 전체가 맞아짐  
> - Vacuous: “조건이 애초에 성립할 일이 없으면” 규칙을 어길 일도 없음


## 7. 재귀적 사고(Recursive Thinking)

재귀는 크게 두 가지가 핵심입니다.
1) **종료 조건(Base Case)**: 어디서 멈출지  
2) **문제 쪼개기**: “현재 문제”를 “더 작은 문제”로 바꿔 호출하기

![](<스크린샷 2026-03-03 153412.png>)


### 예시: 1부터 x까지 더하기

![](<스크린샷 2026-03-03 150444.png>)

- 종료 조건: `x <= 0`이면 0 반환  
- 그 외: `x + sum(x-1)`로 한 단계 줄여서 다시 호출

![](<스크린샷 2026-03-03 153909.png>)

> 쉽게 말하면, “계단을 한 칸씩 내려가면서 숫자를 주워 담고”,  
> 맨 아래(0)에 도착하면 이제부터는 **주워 담은 걸 들고 위로 올라오면서 합치는** 느낌이에요.


In [2]:
def sum_rec(x):
    if x <= 0:
        return 0
    return x + sum_rec(x-1)

for n in range(0, 6):
    print(n, "->", sum_rec(n))


0 -> 0
1 -> 1
2 -> 3
3 -> 6
4 -> 10
5 -> 15


## 8. 점화식과 시간복잡도 감 잡기

점화식은 “한 번 호출될 때 일이 얼마나 생기고, 문제가 얼마나 줄어드는지”를 보는 게 핵심입니다.

### (1) T(n)=T(n-1)+상수 → O(n)
![](<스크린샷 2026-03-03 154250.png>)

### (2) T(n)=T(n-1)+n → O(n²)
![](<스크린샷 2026-03-03 154302.png>)

### (3) T(n)=T(n-1)+log n → O(n log n)
![](<스크린샷 2026-03-03 154317.png>)

### (4) T(n)=T(n/2)+1 → O(log n)
![](<스크린샷 2026-03-03 154353.png>)

### (5) T(n)=T(n/2)+n → O(n)
![](<스크린샷 2026-03-03 154406.png>)

### (6) T(n)=2T(n/2)+n → O(n log n)  (대표: merge sort)
![](<스크린샷 2026-03-03 154419.png>)

### (7) T(n)=3T(n/2)+n → O(n^{log₂ 3})
![](<스크린샷 2026-03-03 154431.png>)

### (8) T(n)=T(n-1)+1/n → O(log n)
![](<스크린샷 2026-03-03 154459.png>)

![](<스크린샷 2026-03-03 151757.png>)


In [4]:
# (감각 확인) 몇 가지 점화식을 실제로 계산해 보기
import math

def rec1(n):
    # T(n)=T(n-1)+1, T(0)=1
    T=[0]*(n+1)
    T[0]=1
    for i in range(1,n+1):
        T[i]=T[i-1]+1
    return T[n]

def rec2(n):
    # T(n)=T(n-1)+n, T(0)=1
    T=[0]*(n+1)
    T[0]=1
    for i in range(1,n+1):
        T[i]=T[i-1]+i
    return T[n]

def rec8(n):
    # T(n)=T(n-1)+1/n, T(1)=1
    T=[0]*(n+1)
    T[1]=1.0
    for i in range(2,n+1):
        T[i]=T[i-1]+1.0/i
    return T[n]

for n in [10, 100, 1000]:
    print("n=", n)
    print("rec1:", rec1(n), " vs n:", n)
    print("rec2:", rec2(n), " vs n^2:", n*n)
    print("rec8:", rec8(n), " vs log n:", math.log(n))
    print()


n= 10
rec1: 11  vs n: 10
rec2: 56  vs n^2: 100
rec8: 2.9289682539682538  vs log n: 2.302585092994046

n= 100
rec1: 101  vs n: 100
rec2: 5051  vs n^2: 10000
rec8: 5.187377517639621  vs log n: 4.605170185988092

n= 1000
rec1: 1001  vs n: 1000
rec2: 500501  vs n^2: 1000000
rec8: 7.485470860550343  vs log n: 6.907755278982137



## 9. 요약

- 논리 문제는 “직관”보다 **진리표/반례**로 정리하면 실수가 크게 줄어든다.  
- 조건명제(p→q)는 **(p=True, q=False)** 한 줄만 거짓이고, **대우(~q→~p)** 가 핵심이다.  
- 재귀는 **종료 조건**이 먼저이며, 점화식은 “일의 양”과 “문제 크기 감소”로 복잡도를 판단한다.

---

## 미니 퀴즈(복습 체크)
1) p→q가 거짓일 때 p와 q의 값은?  
2) 원래 명제와 항상 동치인 것은 역/이/대우 중 무엇?  
3) T(n)=2T(n/2)+n 의 시간복잡도는?  
4) 재귀 함수가 끝나지 않는 가장 흔한 이유는?

(답)
1) p=True, q=False  
2) 대우  
3) O(n log n)  
4) 종료 조건이 없거나, 문제 크기가 줄지 않아서
